# RoadSight: reproduce the small config on a free GPU

Runs on Kaggle (GPU T4 x1 or P100, **Internet on**) or Colab. It downloads and prepares BDD100K 10K (~1.3 GB), trains `configs/small.yaml` (lite ResNet-18 at 640x360, 12 epochs) and evaluates it on val.

To run a fork, change `REPO` to its URL.

In [ ]:
REPO = 'https://github.com/vazovski37/roadsight.git'
!git clone -q $REPO roadsight
%cd roadsight
!pip install -q opencv-python-headless pycocotools pyarrow scipy pyyaml onnx onnxruntime

In [ ]:
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# ~5-10 min: semantic masks from the Berkeley mirror, images + instance bitmasks from Hugging Face
!python tools/prepare_data.py --download --workers 4

In [ ]:
# T4 has no bf16 tensor cores: train in fp32 there (amp only enabled for bf16-capable GPUs)
import torch
amp = 'bf16' if torch.cuda.get_device_capability()[0] >= 8 else 'none'  # T4 = 7.5
!python tools/train.py configs/small.yaml train.amp=$amp data.workers=4

In [ ]:
import json; m = json.load(open('runs/small/metrics.json'))
print({k: round(v, 2) for k, v in m.items() if k.startswith('val/') and '/' not in k[4:]})

In [ ]:
# export the trained model and score the ONNX version on val (CPU)
!python tools/export_onnx.py runs/small --size 640 360 --out models/small.onnx
!python tools/onnx_eval.py models/small.onnx --split val --limit 200